In [ ]:
# --- BƯỚC 0: SETUP FOR RTX PRO 6000 BLACKWELL (KAGGLE OFFLINE) ---
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

OFFLINE_MODE = True
OFFLINE_DATASET_CANDIDATES = [
    "/kaggle/input/datasets/namthi/offline-kaggle-deps/offline_kaggle_deps",
    "/kaggle/input/datasets/namthi/offline-kaggle-deps",
    "/kaggle/input/offline-kaggle-deps/offline_kaggle_deps",
    "/kaggle/input/offline-kaggle-deps",
]

# torch 2.7.0+cu128 - ho tro SM 12.0 (RTX PRO 6000 Blackwell)
TORCH_TARGET_VERSION = "2.7.0"

def _resolve_offline_dataset_root(candidates):
    for base in candidates:
        probe_dirs = [base, os.path.join(base, "offline_kaggle_deps"), os.path.join(base, "offline-kaggle-deps")]
        for d in probe_dirs:
            if not os.path.isdir(d):
                continue
            if os.path.isdir(os.path.join(d, "wheels")) and os.path.isfile(os.path.join(d, "requirements-runtime.txt")):
                return d
    return None

OFFLINE_DATASET_DIR = _resolve_offline_dataset_root(OFFLINE_DATASET_CANDIDATES)
OFFLINE_WHEEL_DIR = os.path.join(OFFLINE_DATASET_DIR, "wheels") if OFFLINE_DATASET_DIR else ""
OFFLINE_REQ_FILE = os.path.join(OFFLINE_DATASET_DIR, "requirements-runtime.txt") if OFFLINE_DATASET_DIR else ""

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

def _current_py_tag():
    return f"cp{sys.version_info.major}{sys.version_info.minor}"

def _wheel_matches_python_tag(filename):
    low = filename.lower()
    return ("py3-none-any" in low) or ("py3-none-manylinux" in low) or (_current_py_tag() in low)

def _find_wheel(package_name, version_prefix=None, require_python_match=False):
    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        return None
    prefix = f"{package_name.lower()}-"
    cands = []
    for fn in os.listdir(OFFLINE_WHEEL_DIR):
        low = fn.lower()
        if not low.endswith(".whl"):
            continue
        if not low.startswith(prefix):
            continue
        if version_prefix and f"-{version_prefix}" not in low:
            continue
        if require_python_match and (not _wheel_matches_python_tag(fn)):
            continue
        cands.append(fn)
    if not cands:
        return None
    return os.path.join(OFFLINE_WHEEL_DIR, sorted(cands)[-1])

if OFFLINE_MODE and not OFFLINE_DATASET_DIR:
    raise FileNotFoundError(f"Offline dataset not found. Checked: {OFFLINE_DATASET_CANDIDATES}")

print(f">>> Using torch target: {TORCH_TARGET_VERSION}")
print(f">>> Runtime Python tag: {_current_py_tag()}")

torch_was_pinned = False

if OFFLINE_MODE:
    print(">>> OFFLINE MODE: install dependencies from dataset")
    print(f">>> OFFLINE_DATASET_DIR = {OFFLINE_DATASET_DIR}")

    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        raise FileNotFoundError(f"Wheels folder not found: {OFFLINE_WHEEL_DIR}")
    if not os.path.isfile(OFFLINE_REQ_FILE):
        raise FileNotFoundError(f"requirements-runtime.txt not found: {OFFLINE_REQ_FILE}")

    # -- 1. Filter requirements: skip packages that should use Kaggle's preinstalled --
    filtered_req = "/kaggle/working/requirements-runtime-filtered.txt"
    # QUAN TRONG: skip numpy/scipy de tranh "numpy.dtype size changed" error
    skip_prefixes = ("pyarrow", "torch", "torchvision", "torchaudio", "triton", "numpy", "scipy")
    with open(OFFLINE_REQ_FILE, "r", encoding="utf-8") as rf, open(filtered_req, "w", encoding="utf-8") as wf:
        for line in rf:
            s = line.strip().lower()
            if (not s) or s.startswith("#"):
                wf.write(line)
                continue
            normalized = s.replace(" ", "")
            if normalized.startswith(skip_prefixes):
                continue
            wf.write(line)

    # -- 2. Install non-torch deps (--no-deps tranh pip resolver conflict) --
    print(">>> Installing non-torch deps (filtered, --no-deps)...")
    _run([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", OFFLINE_WHEEL_DIR,
        "--no-deps",
        "-r", filtered_req,
    ])

    # -- 3. Install nvidia-cusparselt TRUOC torch (provides libcusparseLt.so.0) --
    cusparselt_whls = [
        f for f in os.listdir(OFFLINE_WHEEL_DIR)
        if f.endswith(".whl") and "cusparselt" in f.lower()
    ]
    for whl in cusparselt_whls:
        whl_path = os.path.join(OFFLINE_WHEEL_DIR, whl)
        print(f">>> Installing NVIDIA lib: {whl}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", whl_path])

    # -- 4. Install torch cu128 (SM 12.0 support) --
    torch_whl = _find_wheel("torch", TORCH_TARGET_VERSION, require_python_match=True)
    if not torch_whl:
        available = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.lower().startswith("torch-") and f.endswith(".whl")]
        print(f"WARN: No torch {TORCH_TARGET_VERSION} wheel found. Available: {available}")
        print("WARN: Keep using runtime torch preinstalled by Kaggle.")
    else:
        print(f">>> Installing torch: {os.path.basename(torch_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", torch_whl])
        torch_was_pinned = True

    # -- 5. Install torchvision/torchaudio cu128 --
    tv_whl = _find_wheel("torchvision", require_python_match=True)
    ta_whl = _find_wheel("torchaudio", require_python_match=True)
    if tv_whl:
        print(f">>> Installing torchvision: {os.path.basename(tv_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", tv_whl])
    if ta_whl:
        print(f">>> Installing torchaudio: {os.path.basename(ta_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", ta_whl])

    # -- 6. Install colpali-engine --
    colpali_whls = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.endswith(".whl") and "colpali" in f.lower()]
    if colpali_whls:
        colpali_path = os.path.join(OFFLINE_WHEEL_DIR, sorted(colpali_whls)[0])
        print(f">>> Installing colpali-engine (--no-deps): {os.path.basename(colpali_path)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", OFFLINE_WHEEL_DIR, "--no-deps", colpali_path])
    else:
        raise FileNotFoundError("No colpali-engine wheel found.")

else:
    print(">>> ONLINE MODE")
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow", "pyarrow"])
    _run([sys.executable, "-m", "pip", "install", "pyarrow<20.0.0"])
    _run([sys.executable, "-m", "pip", "install", "-U", "git+https://github.com/illuin-tech/colpali"])
    _run([sys.executable, "-m", "pip", "install", "-U", f"torch=={TORCH_TARGET_VERSION}", "transformers", "accelerate", "peft", "bitsandbytes"])
    _run([sys.executable, "-m", "pip", "install", "-U", "torchvision", "torchaudio"])

# -- Preload NVIDIA libs (torch cu128 can libcusparseLt.so.0) --
import ctypes
import glob

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + (":" + existing_ld if existing_ld else "")
    print(f">>> Added {len(nvidia_lib_dirs)} NVIDIA lib dirs to LD_LIBRARY_PATH")

for pattern in [
    "/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*",
    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/lib*.so*",
]:
    for lib_path in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

cusparselt_found = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*")
if cusparselt_found:
    print(f">>> Preloaded libcusparseLt: {cusparselt_found[0]}")
else:
    print("WARN: libcusparseLt.so not found in nvidia packages!")

import torch

print(f">>> torch version: {torch.__version__}")
if torch_was_pinned and (not torch.__version__.startswith(TORCH_TARGET_VERSION)):
    raise RuntimeError(
        f"Expected torch {TORCH_TARGET_VERSION}, but got {torch.__version__}. "
        "Please verify offline wheel pack and restart kernel."
    )
if not torch_was_pinned:
    print(">>> Using runtime torch (wheel pin skipped).")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    vram_gb = props.total_memory / (1024 ** 3)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f">>> GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | CC={cc_major}.{cc_minor}")
else:
    raise RuntimeError("CUDA is not available.")

PERF_CFG = {
    "GPU_NAME": gpu_name,
    "VRAM_GB": round(vram_gb, 1),
    "BATCH_SIZE_ENCODE": 12,
    "BATCH_Q_EVAL": 12,
    "DOC_CHUNK_SIZE": 1024,
    "SAVE_EVERY": 900,
}
print(
    ">>> RTX6000 config: "
    f"BATCH_SIZE_ENCODE={PERF_CFG['BATCH_SIZE_ENCODE']}, "
    f"BATCH_Q_EVAL={PERF_CFG['BATCH_Q_EVAL']}, "
    f"DOC_CHUNK_SIZE={PERF_CFG['DOC_CHUNK_SIZE']}"
)

print(">>> Setup dependencies done")

In [ ]:
# --- PATCH: Fix colpali_engine import conflict (KEEP paligemma for ColPali) ---
import sys
import types

def _patch_colpali_stub_non_paligemma():
    # Clear cache de tranh trang thai import do dang
    stale = [k for k in list(sys.modules.keys()) if k.startswith("colpali_engine")]
    for k in stale:
        del sys.modules[k]

    # Stub cac family khong dung
    problem_modules = [
        "colpali_engine.models.gemma3",
        "colpali_engine.models.gemma3.bigemma3",
        "colpali_engine.models.gemma3.colgemma3",
        "colpali_engine.models.modernvbert",
        "colpali_engine.models.modernvbert.bivbert",
        "colpali_engine.models.modernvbert.colvbert",
        "colpali_engine.models.qwen2",
        "colpali_engine.models.qwen2.biqwen2",
        "colpali_engine.models.qwen2.colqwen2",
        "colpali_engine.models.qwen3",
        "colpali_engine.models.qwen3.biqwen3",
        "colpali_engine.models.qwen3.colqwen3",
        "colpali_engine.models.qwen3_5",
        "colpali_engine.models.qwen3_5.biqwen3_5",
        "colpali_engine.models.qwen3_5.colqwen3_5",
        "colpali_engine.models.qwen_omni",
        "colpali_engine.models.qwen_omni.colqwen2_5_omni",
    ]

    for name in problem_modules:
        if name not in sys.modules:
            sys.modules[name] = types.ModuleType(name)

    # Cac class duoc colpali_engine.models.__init__ import truc tiep
    stub_class_map = {
        "colpali_engine.models.gemma3": [
            "BiGemma3", "BiGemmaProcessor3", "ColGemma3", "ColGemmaProcessor3"
        ],
        "colpali_engine.models.modernvbert": [
            "BiModernVBert", "BiModernVBertProcessor", "ColModernVBert", "ColModernVBertProcessor"
        ],
        "colpali_engine.models.qwen2": [
            "BiQwen2", "BiQwen2Processor", "ColQwen2", "ColQwen2Processor"
        ],
        "colpali_engine.models.qwen3": [
            "BiQwen3", "BiQwen3Processor", "ColQwen3", "ColQwen3Processor"
        ],
        "colpali_engine.models.qwen3_5": [
            "BiQwen3_5", "BiQwen3_5Processor", "ColQwen3_5", "ColQwen3_5Processor"
        ],
        "colpali_engine.models.qwen_omni": [
            "ColQwen2_5Omni", "ColQwen2_5OmniProcessor"
        ],
    }

    for mod_name, cls_list in stub_class_map.items():
        mod = sys.modules[mod_name]
        for cls_name in cls_list:
            setattr(mod, cls_name, type(cls_name, (), {}))

_patch_colpali_stub_non_paligemma()

from colpali_engine.models.paligemma import ColPali, ColPaliProcessor
print(f"ColPali ready: {ColPali}")

In [ ]:
# --- BƯỚC 0.5: QUICK EXPLORE VIDORE V3 DATASET STRUCTURE ---
from pathlib import Path
from collections import Counter
import json

VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
VIDORE_DOMAIN_CANDIDATES = [
    "vidore_v3",
    "vidore_v3_computer_science",
    "vidore_v3_energy",
    "vidore_v3_finance_en",
    "vidore_v3_hr",
    "vidore_v3_industrial",
    "vidore_v3_pharmaceuticals",
    "vidore_v3_physics",
]

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"Dataset root not found: {VIDORE_DATASET_ROOT}")

print(f">>> Explore ViDoRe root: {VIDORE_DATASET_ROOT}")

resolved_domains = []
for d in VIDORE_DOMAIN_CANDIDATES:
    p = root / d
    if p.exists() and p.is_dir():
        resolved_domains.append(p)

if not resolved_domains:
    resolved_domains = [p for p in sorted(root.iterdir()) if p.is_dir()]

print(f">>> Domains found: {len(resolved_domains)}")
for p in resolved_domains:
    print("   -", p.name)

ext_counter = Counter()
all_files = []
for d in resolved_domains:
    for fp in d.rglob("*"):
        if fp.is_file():
            ext = fp.suffix.lower() or "<no_ext>"
            ext_counter[ext] += 1
            all_files.append(fp)

print("\n>>> File extension stats:")
for ext, cnt in ext_counter.most_common(20):
    print(f"   {ext}: {cnt}")

print("\n>>> Sample files:")
for fp in all_files[:25]:
    print("   -", fp.relative_to(root))

preview_candidates = [fp for fp in all_files if fp.suffix.lower() in {".parquet", ".json", ".jsonl"}]
print(f"\n>>> Preview candidates: {len(preview_candidates)}")

def _preview_file(fp):
    print(f"\n=== PREVIEW: {fp.relative_to(root)} ===")
    sfx = fp.suffix.lower()
    try:
        if sfx == ".parquet":
            import pandas as pd
            df = pd.read_parquet(fp)
            print("shape:", df.shape)
            print("columns:", list(df.columns))
            if len(df) > 0:
                print("sample row keys:", list(df.iloc[0].to_dict().keys()))
        elif sfx == ".jsonl":
            with open(fp, "r", encoding="utf-8") as f:
                line = f.readline().strip()
            obj = json.loads(line) if line else {}
            print("first record keys:", list(obj.keys()) if isinstance(obj, dict) else type(obj))
        elif sfx == ".json":
            with open(fp, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, list) and len(obj) > 0 and isinstance(obj[0], dict):
                print("json is list; first record keys:", list(obj[0].keys()))
            elif isinstance(obj, dict):
                print("json top-level keys:", list(obj.keys()))
            else:
                print("json type:", type(obj))
    except Exception as e:
        print("preview error:", e)

for fp in preview_candidates[:5]:
    _preview_file(fp)

globals()["VIDORE_DATASET_ROOT"] = VIDORE_DATASET_ROOT
globals()["VIDORE_DOMAIN_PATHS"] = [str(p) for p in resolved_domains]
print("\n>>> Explore done. Use output above to fine-tune mapping if needed.")

In [ ]:
# --- BƯỚC 1: SETUP DATA (STRICT PAGE-LEVEL FROM VIDORE V3 CORPUS) ---
import os
import io
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

# Encode full dataset (no document slicing limit)
START_IDX = 0
END_IDX = None

# Neu muon chi encode mot vai domain, dat list ten domain tai day. De [] de lay tat ca.
DOMAIN_FILTER = []

VIDORE_DATASET_ROOT = globals().get("VIDORE_DATASET_ROOT", "/kaggle/input/datasets/namthi/vidore-v3")
VIDORE_DOMAIN_CANDIDATES = [
    "vidore_v3_computer_science",
    "vidore_v3_energy",
    "vidore_v3_finance_en",
    "vidore_v3_hr",
    "vidore_v3_industrial",
    "vidore_v3_pharmaceuticals",
    "vidore_v3_physics",
]

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"Khong tim thay dataset root: {VIDORE_DATASET_ROOT}")

# Resolve domain folders
candidate_dirs = []
for d in VIDORE_DOMAIN_CANDIDATES:
    p = root / d
    if p.is_dir():
        candidate_dirs.append(p)

if not candidate_dirs:
    candidate_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]

if DOMAIN_FILTER:
    domain_dirs = [p for p in candidate_dirs if p.name in set(DOMAIN_FILTER)]
else:
    domain_dirs = candidate_dirs

if not domain_dirs:
    raise ValueError("Khong tim thay domain nao hop le de encode")

print(f">>> VIDORE root: {root}")
print(f">>> Domains selected ({len(domain_dirs)}): {[p.name for p in domain_dirs]}")


def _resolve_img_path(path_value, domain_dir):
    p = str(path_value).strip()
    if not p:
        return None

    if os.path.isfile(p):
        return p

    cand1 = domain_dir / p
    if cand1.is_file():
        return str(cand1)

    cand2 = root / p
    if cand2.is_file():
        return str(cand2)

    return None


def _extract_img_type_and_data(image_value, domain_dir):
    # HuggingFace Image feature trong parquet thuong la dict {'bytes':..., 'path':...}
    if image_value is None or (isinstance(image_value, float) and np.isnan(image_value)):
        return None, None

    if isinstance(image_value, dict):
        img_bytes = image_value.get("bytes")
        img_path = image_value.get("path")

        if isinstance(img_bytes, (bytes, bytearray)) and len(img_bytes) > 0:
            return "binary", bytes(img_bytes)

        if img_path is not None:
            resolved = _resolve_img_path(img_path, domain_dir)
            if resolved is not None:
                return "path", resolved

        return None, None

    if isinstance(image_value, (bytes, bytearray)) and len(image_value) > 0:
        return "binary", bytes(image_value)

    if isinstance(image_value, str):
        resolved = _resolve_img_path(image_value, domain_dir)
        if resolved is not None:
            return "path", resolved
        return None, None

    if isinstance(image_value, Image.Image):
        bio = io.BytesIO()
        image_value.convert("RGB").save(bio, format="PNG")
        return "binary", bio.getvalue()

    return None, None


def _iter_corpus_parquets(domain_dir):
    # ViDoRe V3: du lieu page-level nam trong thu muc corpus/*.parquet
    for fp in domain_dir.rglob("corpus/*.parquet"):
        if fp.is_file():
            yield fp


required_columns = {"image", "doc_id", "markdown", "page_number_in_doc"}
normalized_rows = []
domain_stats = {}

for domain_dir in domain_dirs:
    domain = domain_dir.name
    corpus_files = sorted(list(_iter_corpus_parquets(domain_dir)))

    if not corpus_files:
        print(f"WARN: Domain {domain} khong tim thay corpus/*.parquet")
        continue

    scanned_rows = 0
    kept_rows = 0

    print(f"\n>>> Scanning domain: {domain} | corpus_files={len(corpus_files)}")

    for fp in corpus_files:
        try:
            df = pd.read_parquet(fp)
        except Exception as e:
            print(f"Skip file (read error): {fp} | {e}")
            continue

        if df is None or df.empty:
            continue

        missing = required_columns.difference(set(df.columns))
        if missing:
            print(f"Skip file (missing columns {missing}): {fp}")
            continue

        scanned_rows += len(df)

        for ridx, row in df.iterrows():
            doc_name = str(row.get("doc_id", "")).strip() or f"{domain}_{fp.stem}"
            safe_page_raw = row.get("page_number_in_doc", ridx + 1)
            try:
                safe_page = int(float(safe_page_raw))
            except Exception:
                safe_page = int(ridx) + 1

            text_val = row.get("markdown", "")
            final_text = str(text_val).strip() if pd.notna(text_val) else ""
            if not final_text:
                final_text = "Document page."
            final_text = final_text[:8000]

            img_type, img_data = _extract_img_type_and_data(row.get("image"), domain_dir)
            if img_data is None:
                continue

            corpus_id = str(row.get("corpus_id", "")).strip()
            if corpus_id:
                page_key = f"{domain}__{doc_name}__p{safe_page}__{corpus_id}"
            else:
                page_key = f"{domain}__{doc_name}__p{safe_page}"

            normalized_rows.append({
                "domain": domain,
                "join_doc_name": doc_name,
                "safe_page": int(safe_page),
                "img_type": img_type,
                "img_data": img_data,
                "final_text": final_text,
                "page_key": page_key,
                "n_layouts": 1,
                "corpus_id": corpus_id,
            })
            kept_rows += 1

    domain_stats[domain] = {
        "scanned_rows": scanned_rows,
        "kept_rows": kept_rows,
    }

if not normalized_rows:
    raise ValueError("Khong tao duoc page rows tu ViDoRe corpus parquet")

pages_df = pd.DataFrame(normalized_rows)
pages_df = pages_df.drop_duplicates(subset=["page_key"], keep="first").copy()
pages_df = pages_df.sort_values(by=["domain", "join_doc_name", "safe_page", "page_key"]).reset_index(drop=True)

all_docs = sorted(pages_df["join_doc_name"].astype(str).unique().tolist())
target_doc_names = all_docs[START_IDX:END_IDX]
if len(target_doc_names) == 0:
    raise ValueError("Batch rong, kiem tra START_IDX/END_IDX")

sample_pages_df = pages_df[pages_df["join_doc_name"].isin(target_doc_names)].copy()
if sample_pages_df.empty:
    raise ValueError("Khong co page nao sau khi loc theo batch doc")

sample_layouts_df = sample_pages_df.copy()
page_to_layout_indices = {k: [i] for i, k in enumerate(sample_pages_df["page_key"].tolist())}
page_key_to_index = {k: i for i, k in enumerate(sample_pages_df["page_key"].tolist())}

globals()["sample_layouts_df"] = sample_layouts_df
globals()["sample_pages_df"] = sample_pages_df
globals()["page_to_layout_indices"] = page_to_layout_indices
globals()["page_key_to_index"] = page_key_to_index

doc_page_counts = sample_pages_df.groupby("join_doc_name").size().to_dict()

print("=" * 70)
print("VIDORE V3 CORPUS PAGE-LEVEL READY")
print(f"PROCESSING BATCH [{START_IDX}:{END_IDX}]")
print(f"Total docs available: {len(all_docs)}")
print(f"Selected docs: {len(target_doc_names)}")
print(f"Total page rows kept (after batch): {len(sample_pages_df)}")
print("Domain stats:")
for d, st in domain_stats.items():
    print(f"- {d}: scanned_rows={st['scanned_rows']}, kept_rows={st['kept_rows']}")
print(f"Pages per selected doc: {doc_page_counts}")
print("=" * 70)

In [ ]:
# --- BƯỚC 2: LOAD MODEL (ColPali v1.3) ---
print(">>> BƯỚC 2: Loading ColPali v1.3...")

import os
import gc
import torch
import transformers
from peft import PeftModel, PeftConfig

# --- COMPAT PATCH: colpali_engine x transformers>=4.51 (PaliGemma khong con .model) ---
try:
    from transformers import PaliGemmaForConditionalGeneration

    # colpali_engine co goi model.model.language_model.
    # transformers moi doi structure, nen them alias .model -> self.
    if not hasattr(PaliGemmaForConditionalGeneration, "model"):
        PaliGemmaForConditionalGeneration.model = property(lambda self: self)
        print(">>> Applied class-level compat patch: PaliGemmaForConditionalGeneration.model -> self")
except Exception as e:
    print(f">>> WARN: compat patch skipped: {e}")

from colpali_engine.models.paligemma import ColPali, ColPaliProcessor

# Dọn dẹp bộ nhớ trước khi load
gc.collect()
torch.cuda.empty_cache()

print(f">>> transformers version: {transformers.__version__}")

# Thiết lập thiết bị và kiểu dữ liệu (Dtype)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    if major >= 8:
        RUNTIME_DTYPE = torch.bfloat16
    else:
        RUNTIME_DTYPE = torch.float16
    print(f">>> GPU: {gpu_name} | CC={major}.{minor} | Runtime dtype={RUNTIME_DTYPE}")
else:
    RUNTIME_DTYPE = torch.float32

# --- HÀM HỖ TRỢ ---
def _first_existing_path(candidates):
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None

# Đường dẫn Dataset trên Kaggle
COLPALI_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/namthi/colpali-v1-3",
    "/kaggle/input/datasets/namthi/colpali-v1-3/colpali-v1.3",
    "/kaggle/input/colpali-v1-3",
    "/kaggle/input/colpali-v1.3",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/namthi/paligemma-3b-pt-448",
    "/kaggle/input/paligemma-3b-pt-448",
]

model_dir = _first_existing_path(COLPALI_MODEL_CANDIDATES)
base_dir = _first_existing_path(BASE_MODEL_CANDIDATES)

if not model_dir:
    raise FileNotFoundError("Khong tim thay adapter model colpali-v1-3")

# --- LOGIC LOAD MODEL ---
# Kiểm tra xem folder có chứa full weights (đã merge) hay chỉ có adapter
has_full_weight = any(os.path.exists(os.path.join(model_dir, f))
                     for f in ["model.safetensors", "pytorch_model.bin"]) or \
                  any(f.startswith("model-") for f in os.listdir(model_dir))

if has_full_weight:
    print(">>> Detected full weights. Loading directly...")
    model = ColPali.from_pretrained(
        model_dir,
        torch_dtype=RUNTIME_DTYPE,
        device_map="auto" if device == "cuda" else "cpu",
        attn_implementation="eager",
        local_files_only=True,
    )
    processor = ColPaliProcessor.from_pretrained(model_dir, use_fast=True)
else:
    print(">>> Detected adapter-style. Loading Base + Adapter...")
    if not base_dir:
        raise FileNotFoundError("Khong tim thay base model paligemma-3b-pt-448 để nạp adapter")

    # 1. Load Base Model với class ColPali
    base_model = ColPali.from_pretrained(
        base_dir,
        torch_dtype=RUNTIME_DTYPE,
        device_map="auto" if device == "cuda" else "cpu",
        attn_implementation="eager",
        local_files_only=True,
    )

    # 2. Load Adapter LoRA
    model = PeftModel.from_pretrained(
        base_model,
        model_dir,
        local_files_only=True,
    )

    # 3. Merge trọng số để tối ưu tốc độ Inference
    print(">>> Merging weights...")
    model = model.merge_and_unload()

    # 4. Load Processor từ model_dir
    processor = ColPaliProcessor.from_pretrained(model_dir, use_fast=True)

# Chế độ Eval
model = model.eval()

# --- FIX COMPATIBILITY PATCH (nhe, an toan) ---
try:
    # Neu model sau merge khong co .model thi dat alias don gian de tuong thich code cu
    if not hasattr(model, "model"):
        model.model = model
        print(">>> Applied runtime compat patch: model.model -> model")
except Exception as e:
    print(f">>> Patch warning: {e}")

print(">>> [SUCCESS] Model & Processor ready (ColPali v1.3)")

In [ ]:
# --- BƯỚC 3: FUSED INDEXING (PAGE-LEVEL READY-TO-SEARCH) ---
print(">>> BƯỚC 3: Creating page-level index (ColPali v1.3)")

import io
import os
import gc
import pickle
import torch
from PIL import Image
from tqdm.notebook import tqdm

WORKING_DIR = "/kaggle/working"
INDEX_SUFFIX = "vidore_v3_pagelevel"
FINAL_INDEX_PATH = os.path.join(WORKING_DIR, f"colpali13_page_index_{INDEX_SUFFIX}.pkl")
CHECKPOINT_PATH = os.path.join(WORKING_DIR, f"colpali13_checkpoint_{INDEX_SUFFIX}.pkl")

BATCH_SIZE = int(globals().get("PERF_CFG", {}).get("BATCH_SIZE_ENCODE", 12))
SAVE_EVERY = int(globals().get("PERF_CFG", {}).get("SAVE_EVERY", 900))
print(f"Using BATCH_SIZE={BATCH_SIZE}, SAVE_EVERY={SAVE_EVERY}")

fused_index = []
start_idx = 0
skip_encoding = False
cpu_fallback_activated = False

if os.path.exists(FINAL_INDEX_PATH):
    print(f"Found completed index: {FINAL_INDEX_PATH}")
    with open(FINAL_INDEX_PATH, "rb") as f:
        loaded = pickle.load(f)
    if isinstance(loaded, dict) and "fused_index" in loaded:
        fused_index = loaded["fused_index"]
    elif isinstance(loaded, dict) and "embeddings" in loaded:
        fused_index = loaded["embeddings"]
    else:
        fused_index = loaded
    if len(fused_index) > 0:
        print(f"Loaded {len(fused_index)} pages. Skip encoding.")
        skip_encoding = True

elif os.path.exists(CHECKPOINT_PATH):
    print("Found checkpoint. Resuming...")
    with open(CHECKPOINT_PATH, "rb") as f:
        ckpt = pickle.load(f)
    fused_index = ckpt.get("fused_index", [])
    start_idx = int(ckpt.get("next_row", len(fused_index)))
    print(f"Restored {len(fused_index)} pages. Continue from row {start_idx}.")
else:
    print("Fresh run.")

if not skip_encoding:
    if "sample_pages_df" not in globals():
        raise ValueError("sample_pages_df not found. Run BƯỚC 1 first.")

    total_pages = len(sample_pages_df)
    print(f"Total pages to process: {total_pages}")
    remaining_df = sample_pages_df.iloc[start_idx:].reset_index(drop=True)

    def load_img_safe(row):
        try:
            if row["img_type"] == "path":
                img = Image.open(row["img_data"]).convert("RGB")
            elif row["img_type"] == "binary":
                img = Image.open(io.BytesIO(row["img_data"])).convert("RGB")
            else:
                img = Image.new("RGB", (224, 224), "white")

            if img.width < 14 or img.height < 14:
                return Image.new("RGB", (224, 224), "white")
            return img
        except Exception:
            return Image.new("RGB", (224, 224), "white")

    _autocast_dtype = globals().get("AUTOCAST_DTYPE", torch.bfloat16)

    def encode_batch(batch_imgs, batch_texts, run_device):
        if run_device == "cuda":
            ctx = torch.autocast(device_type="cuda", dtype=_autocast_dtype, enabled=True)
        else:
            class _DummyCtx:
                def __enter__(self):
                    return None
                def __exit__(self, exc_type, exc, tb):
                    return False
            ctx = _DummyCtx()

        with ctx:
            vis_inputs = processor.process_images(batch_imgs).to(run_device)
            vis_out = model(**vis_inputs)
            vis_list = list(torch.unbind(vis_out.float().cpu()))

            txt_inputs = processor.process_queries(batch_texts, max_length=512, suffix="").to(run_device)
            txt_out = model(**txt_inputs)
            txt_list = list(torch.unbind(txt_out.float().cpu()))

        for i in range(len(vis_list)):
            fused_vec = torch.cat([vis_list[i], txt_list[i]], dim=0).to(torch.float16).contiguous()
            fused_index.append(fused_vec)

    batch_imgs, batch_texts = [], []

    for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Encoding pages"):
        current_real_idx = start_idx + i
        batch_imgs.append(load_img_safe(row))
        batch_texts.append(str(row["final_text"]).replace("<image>", " ")[:4000])

        if len(batch_imgs) >= BATCH_SIZE or i == len(remaining_df) - 1:
            with torch.no_grad():
                try:
                    encode_batch(batch_imgs, batch_texts, device)
                except Exception as e:
                    err = str(e).lower()
                    is_cuda_err = ("cuda error" in err) or ("no kernel image is available" in err) or ("out of memory" in err)

                    if is_cuda_err and not cpu_fallback_activated:
                        print(f"CUDA batch error: {e}")
                        print("Switching to CPU fallback...")
                        model = model.to("cpu")
                        device = "cpu"
                        globals()["device"] = "cpu"
                        cpu_fallback_activated = True
                        torch.cuda.empty_cache()
                        gc.collect()
                        try:
                            encode_batch(batch_imgs, batch_texts, "cpu")
                        except Exception as e2:
                            print(f"CPU fallback failed: {e2}. Skip batch.")
                    else:
                        print(f"Batch {current_real_idx} error: {e}. Skip batch.")

            batch_imgs, batch_texts = [], []

            if len(fused_index) > 0 and len(fused_index) % SAVE_EVERY == 0:
                with open(CHECKPOINT_PATH, "wb") as f:
                    pickle.dump({"fused_index": fused_index, "next_row": current_real_idx + 1}, f)
                print(f"Checkpoint saved: {len(fused_index)} pages")

# Save ready-to-search payload
if "sample_pages_df" in globals():
    page_keys = sample_pages_df["page_key"].tolist()
    meta_cols = ["domain", "join_doc_name", "safe_page", "page_key", "n_layouts"]
    if "corpus_id" in sample_pages_df.columns:
        meta_cols.append("corpus_id")

    payload = {
        "model_name": "vidore/colpali-v1.3",
        "dataset": "vidore-v3",
        "index_level": "page",
        "num_pages": len(fused_index),
        "fused_index": fused_index,
        "page_keys": page_keys[:len(fused_index)],
        "page_key_to_row": {k: i for i, k in enumerate(page_keys[:len(fused_index)])},
        "metadata": sample_pages_df.iloc[:len(fused_index)][meta_cols].to_dict("records"),
    }
else:
    payload = {
        "model_name": "vidore/colpali-v1.3",
        "dataset": "vidore-v3",
        "fused_index": fused_index,
    }

with open(FINAL_INDEX_PATH, "wb") as f:
    pickle.dump(payload, f)

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

if cpu_fallback_activated:
    print(">>> CPU fallback was used.")

if "sample_pages_df" in globals() and len(fused_index) != len(sample_pages_df):
    print(f"WARNING: index size ({len(fused_index)}) != page rows ({len(sample_pages_df)})")

print("=" * 60)
print("BƯỚC 3 HOÀN TẤT (PAGE-LEVEL, VIDORE V3)")
print(f"Total pages in index: {len(fused_index)}")
print(f"Index path:           {FINAL_INDEX_PATH}")
print("=" * 60)